# Criação do DW ETL — ITBI Recife

Cria o schema `dw_etl` e suas tabelas no Supabase.

> O schema `staging` é criado automaticamente pelo ELT.ipynb.
> O schema `dw_elt` é criado automaticamente pelo dbt.

In [ ]:
%pip install psycopg2-binary sqlalchemy python-dotenv --quiet

In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

load_dotenv(os.path.join(os.path.dirname(os.path.abspath('__file__')), '.env'), override=True)

usuario_db    = os.getenv('DB_USER')
senha_db      = os.getenv('DB_PASSWORD')
host_db       = os.getenv('DB_HOST')
porta_db      = int(os.getenv('DB_PORT'))
nome_banco_db = os.getenv('DB_NAME')

faltando = [k for k, v in {'DB_USER': usuario_db, 'DB_PASSWORD': senha_db, 'DB_HOST': host_db, 'DB_PORT': porta_db, 'DB_NAME': nome_banco_db}.items() if not v]
if faltando:
    raise ValueError(f'Variáveis ausentes no .env: {faltando}')

engine = create_engine(URL.create(
    drivername="postgresql+psycopg2",
    username=usuario_db,
    password=senha_db,
    host=host_db,
    port=porta_db,
    database=nome_banco_db,
    query={"client_encoding": "utf8"}
))

with engine.connect() as conn:
    print("Conectado:", conn.execute(text("SELECT version()")).scalar()[:50])

In [ ]:
DDL = """
CREATE SCHEMA IF NOT EXISTS dw_etl;

CREATE TABLE IF NOT EXISTS dw_etl.dim_tempo (
    sk_tempo        SERIAL PRIMARY KEY,
    data_completa   DATE,
    dia             INTEGER,
    mes             INTEGER,
    nome_mes        VARCHAR(20),
    trimestre       INTEGER,
    ano             INTEGER,
    dia_semana      INTEGER,
    nome_dia_semana VARCHAR(20)
);

CREATE TABLE IF NOT EXISTS dw_etl.dim_localizacao (
    sk_localizacao  SERIAL PRIMARY KEY,
    cod_logradouro  VARCHAR(50),
    logradouro      VARCHAR(255),
    numero          VARCHAR(50),
    bairro          VARCHAR(150)
);

CREATE TABLE IF NOT EXISTS dw_etl.dim_imovel (
    sk_imovel           SERIAL PRIMARY KEY,
    complemento_numero  VARCHAR(255),
    nome_edificio       VARCHAR(255),
    ano_construcao      INTEGER,
    decada_construcao   INTEGER
);

CREATE TABLE IF NOT EXISTS dw_etl.dim_caracteristica (
    sk_caracteristica   SERIAL PRIMARY KEY,
    tipo_imovel         VARCHAR(100),
    tipo_construcao     VARCHAR(100),
    padrao_acabamento   VARCHAR(100),
    estado_conservacao  VARCHAR(100),
    tipo_ocupacao       VARCHAR(100)
);

CREATE TABLE IF NOT EXISTS dw_etl.fato_transacao_itbi (
    sk_transacao        BIGSERIAL PRIMARY KEY,
    sk_tempo            INTEGER REFERENCES dw_etl.dim_tempo(sk_tempo),
    sk_localizacao      INTEGER REFERENCES dw_etl.dim_localizacao(sk_localizacao),
    sk_imovel           INTEGER REFERENCES dw_etl.dim_imovel(sk_imovel),
    sk_caracteristica   INTEGER REFERENCES dw_etl.dim_caracteristica(sk_caracteristica),
    valor_avaliacao     NUMERIC(15,2),
    area_terreno        NUMERIC(12,2),
    area_construida     NUMERIC(12,2),
    fracao_ideal        NUMERIC(10,6),
    sfh                 NUMERIC(15,2),
    valor_m2_construido NUMERIC(15,2),
    idade_imovel_anos   INTEGER
);
"""

with engine.connect() as conn:
    conn.execute(text(DDL))
    conn.commit()

print("Schema dw_etl criado com sucesso!")
print("  dw_etl → dim_tempo, dim_localizacao, dim_imovel, dim_caracteristica, fato_transacao_itbi")